# GitHub API Client

An example of how to use the GitHub Client implemented in `src/gh_signals/gh_client.py` to fetch live data. It handles authentication (authenticating with your GitHub account gives you much higher rate limits). It also handles rate limits gracefully.

The way the client authenticates with the first token it finds: one passed to `GitHubClient`, then `GITHUB_TOKEN` set as an environment variable, then `gh auth token` from the GitHub CLI. Without any of those it still works with lower rate limits.

By default responses are cached under `.cache/github/` for a day, so re-running a cell does not spend more of the quota.

In [1]:
import asyncio
import csv
from pathlib import Path
import random

from gh_signals.github_client import GitHubClient

## Load Repo List

`data/good_repos.csv` holds repositories we treat as the positive class and `data/uncertain.csv` as the negative class.
Let's sample a few of the good repos and get their number of stars and forks.

In [2]:
csv_path = Path("..", "data", "good_repos.csv")
with csv_path.open() as file:
    good_repos = [row["repo"] for row in csv.DictReader(file)]

chosen = random.sample(good_repos, 3)
chosen

['numpy/numpy', 'microsoft/azure-pipelines-agent', 'microsoft/onnxruntime']

## Fetch Data Using the GitHub Client

In [3]:
async with GitHubClient() as client:
    # Fetch the repository information for each chosen repository concurrently
    results = await asyncio.gather(*(client.get_repo(name) for name in chosen))

for result in results:
    print(f"{result.full_name}: {result.stargazers_count} stars, {result.forks_count} forks")

numpy/numpy: 32530 stars, 12630 forks
microsoft/azure-pipelines-agent: 1909 stars, 929 forks
microsoft/onnxruntime: 21369 stars, 4124 forks
